In [ ]:
import sys
sys.path.insert(0, "/Users/shreyessreedhar/Prediction Markets Project/ACM-AI-Spring-2026-Prediction-Markets")

import pandas as pd
from extensions.sports_reversion.strategy import compute_signals

raw = pd.read_csv("/Users/shreyessreedhar/Downloads/nba_elo.csv")

elo = raw[
    (raw["_iscopy"] == 0) &
    (raw["is_playoffs"] == 0) &
    (raw["year_id"] >= 2010)
].copy()

elo = elo.rename(columns={
    "date_game": "date",
    "team_id": "team1",
    "opp_id": "team2",
    "forecast": "elo_prob1",
    "pts": "score1",
    "opp_pts": "score2",
})

elo["date"] = pd.to_datetime(elo["date"])
elo = elo.dropna(subset=["score1", "score2", "elo_prob1"])

print(f"Total games loaded: {len(elo)}")
print(f"Date range: {elo['date'].min()} to {elo['date'].max()}")

signals = compute_signals(elo)
print(f"\nTotal signals: {len(signals)}")
signals.head()

In [ ]:
from execution.kelly import kelly_fraction, dollars_to_contracts
from backtest.metrics import print_metrics

STARTING_BALANCE = 1000.0
balance = STARTING_BALANCE
cumulative_pnl = 0.0
trades = []

for _, row in signals.iterrows():
    bet_dollars, side = kelly_fraction(
        p_model=row["p_model"],
        market_price=row["market_price"],
        bankroll=balance,
        kelly_multiplier=0.25,
        max_position_pct=0.05,
    )
    if bet_dollars <= 0:
        continue

    price = row["market_price"] if side == "YES" else (1 - row["market_price"])
    n_contracts = dollars_to_contracts(bet_dollars, price)
    if n_contracts == 0:
        continue

    cost = n_contracts * price
    won = (side == "YES" and row["resolved_yes"] == 1) or \
          (side == "NO" and row["resolved_yes"] == 0)
    pnl = n_contracts * (1 - price) if won else -cost
    balance += pnl
    cumulative_pnl += pnl

    trades.append({
        "p_model": row["p_model"],
        "market_price": row["market_price"],
        "side": side,
        "n_contracts": n_contracts,
        "cost": round(cost, 4),
        "edge": round(abs(row["p_model"] - row["market_price"]), 4),
        "resolved_yes": row["resolved_yes"],
        "won": won,
        "pnl": round(pnl, 4),
        "balance": round(balance, 2),
        "cumulative_pnl": round(cumulative_pnl, 4),
    })

trades_df = pd.DataFrame(trades)
print_metrics(trades_df, starting_balance=STARTING_BALANCE)

In [ ]:
trades_df["cumulative_pnl"].plot(title="Sports Reversion Cumulative P&L", ylabel="P&L ($)", xlabel="Trade #")